# Longitudinal Mining Performance

This notebook computes the workload and runtime tables reported for the 29-library longitudinal study.


In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
BOOL_COLUMNS = ["is_merge_commit", "api_changed", "has_java_changes", "has_pom_changes"]
TIMING_COLUMNS = ["checkout_time_ms", "api_time_ms", "diff_time_ms"]

frames = [pd.read_csv(path, low_memory=False) for path in sorted(DATA_DIR.glob("*-commits.csv"))]
commits = pd.concat(frames, ignore_index=True)
for column in BOOL_COLUMNS:
    commits[column] = commits[column].astype(str).str.lower().eq("true")
for column in TIMING_COLUMNS:
    commits[column] = pd.to_numeric(commits[column], errors="coerce").fillna(0)

commits["analysis_time_ms"] = commits[TIMING_COLUMNS].sum(axis=1)
commits["category"] = "Non-Java"
commits.loc[commits["has_java_changes"], "category"] = "Java, API stable"
commits.loc[commits["api_changed"], "category"] = "Java, API changed"


In [ ]:
category_order = ["Non-Java", "Java, API stable", "Java, API changed"]
java_commits = commits["has_java_changes"].sum()

workload_table = (
    commits.groupby("category", sort=False)
    .agg(
        Count=("category", "size"),
        Checkout=("checkout_time_ms", lambda values: values[values > 0].median()),
        API=("api_time_ms", lambda values: values[values > 0].median()),
        Diff=("diff_time_ms", lambda values: values[values > 0].median()),
    )
    .reindex(category_order)
)
workload_table["All (%)"] = 100 * workload_table["Count"] / len(commits)
workload_table["Java (%)"] = [pd.NA, 100 * workload_table.loc["Java, API stable", "Count"] / java_commits, 100 * workload_table.loc["Java, API changed", "Count"] / java_commits]
workload_table = workload_table[["Count", "All (%)", "Java (%)", "Checkout", "API", "Diff"]]

total_analysis_hours = commits["analysis_time_ms"].sum() / 3_600_000
runtime_summary = pd.DataFrame(
    {
        "metric": [
            "libraries",
            "commits",
            "total analysis time (hours)",
            "mean analysis time (ms per commit)",
            "throughput (commits per second)",
            "checkout share (%)",
            "API construction share (%)",
            "API differencing share (%)",
        ],
        "value": [
            commits["library"].nunique(),
            len(commits),
            total_analysis_hours,
            commits["analysis_time_ms"].mean(),
            len(commits) / (total_analysis_hours * 3600),
            100 * commits["checkout_time_ms"].sum() / commits["analysis_time_ms"].sum(),
            100 * commits["api_time_ms"].sum() / commits["analysis_time_ms"].sum(),
            100 * commits["diff_time_ms"].sum() / commits["analysis_time_ms"].sum(),
        ],
    }
)

display(workload_table.round(1))
display(runtime_summary.round(2))


In [ ]:
def library_runtime(group):
    api_changed = group[group["api_changed"]]
    return pd.Series(
        {
            "commits": len(group),
            "total_analysis_minutes": group["analysis_time_ms"].sum() / 60_000,
            "median_api_changing_commit_ms": api_changed["analysis_time_ms"].median(),
        }
    )

library_runtime_table = (
    commits.groupby("library")
    .apply(library_runtime, include_groups=False)
    .sort_values("total_analysis_minutes", ascending=False)
)
display(library_runtime_table.round(1))
